In [1]:
import xarray as xr

In [3]:
ds = xr.open_dataset("/home/nannau/linep/LineP-team-holdsworth_CTD_stations.nc")
ds.lon.values

array([233.34736454, 232.91256695, 232.3473954 , 231.82131345,
       231.25477052, 230.81615428, 230.37668469, 229.80956384,
       229.36891704, 228.26405229, 227.37706324, 226.26484991,
       225.37259645, 224.37510621, 223.25311263, 222.35398817,
       221.34524781, 220.33245248, 219.31595507, 218.29611663,
       217.38882423, 216.35802324, 215.67520845, 214.99289791])

In [27]:
# check that the dimensions are correct
correct_dimensions = {
    "time": 623,
    "station": 24,
    "PRESSURE_BIN_CNTR": 4000,
}

required_vars = [
    "TEMPERATURE",
    "SALINITY",
]

alternative_groups = [
    ["OXYGEN_UMOL_KG", "OXYGEN_MMOL_M3"],
]

required_coords = {
    "lat": "station",
    "lon": "station",
}

expected_units = {
    "SALINITY": ["psu"],
    "salinity": ["psu"],
    
    "OXYGEN_UMOL_KG": ["umol/kg", "µmol/kg"],
    "OXYGEN_MMOL_M3": ["mmol/m3", "mmol/m³"],
    
    "lat": ["degrees", "degree_north", "deg"],
    "lon": ["degrees", "degree_east", "deg"],
    
    "PRESSURE_BIN_CNTR": ["dbar"],
    
    # time ignored
    # station ignored
}

def validate_dims(ds, expected_dims, *, name="dataset", raise_error=True):
    """
    Validate that an xarray Dataset or DataArray matches a set of expected
    dimension names and sizes.

    Parameters
    ----------
    ds : xr.Dataset or xr.DataArray
        The object whose dims to check.
    expected_dims : dict
        Mapping of dim name -> expected size.
    name : str
        Name used in error messages.
    raise_error : bool
        If True, raises a ValueError on problems. If False, only returns a report.

    Returns
    -------
    dict with:
        - missing_dims
        - unexpected_dims
        - size_mismatches
        - identical (bool)
    """
    actual_dims = dict(ds.sizes)

    # Expected but missing
    missing = {
        d: expected_dims[d]
        for d in expected_dims
        if d not in actual_dims
    }

    # Present but not expected
    unexpected = {
        d: actual_dims[d]
        for d in actual_dims
        if d not in expected_dims
    }

    # Size mismatches for shared dims
    size_mismatches = {
        d: (actual_dims[d], expected_dims[d])
        for d in expected_dims
        if d in actual_dims and actual_dims[d] != expected_dims[d]
    }

    identical = not missing and not unexpected and not size_mismatches

    report = {
        "missing_dims": missing,
        "unexpected_dims": unexpected,
        "size_mismatches": size_mismatches,
        "identical": identical,
    }

    # Build human-readable report
    if raise_error and not identical:
        lines = [f"Dimension mismatch in {name}:"]
        if missing:
            lines.append("  ❌ Missing expected dimensions:")
            lines.extend([f"      - {d}: expected size {sz}" for d, sz in missing.items()])
        if unexpected:
            lines.append("  ❌ Unexpected dimensions present:")
            lines.extend([f"      - {d}: actual size {sz}" for d, sz in unexpected.items()])
        if size_mismatches:
            lines.append("  ❌ Dimensions with incorrect sizes:")
            lines.extend([f"      - {d}: actual {act}, expected {exp}" for d, (act, exp) in size_mismatches.items()])

        raise ValueError("\n".join(lines))
    print(f"Dimension validation for {name}: {'✅ PASSED' if identical else '❌ FAILED'}")
    return report

def validate_vars(
    ds,
    required_vars=None,
    *,
    alternative_groups=None,
    name="dataset",
    raise_error=True
):
    """
    Validate that required variables exist in an xarray Dataset, including
    support for groups where at least one variable is required.

    Parameters
    ----------
    ds : xr.Dataset
        Dataset being validated.
    required_vars : list of str
        Variables that must be present.
    alternative_groups : list of lists
        Each inner list is a group of acceptable alternatives.
    name : str
        Dataset name for error messages.
    raise_error : bool
        Whether to raise ValueError when errors are found.

    Returns
    -------
    dict with:
        - missing_vars
        - missing_groups
        - identical (bool)
    """
    required_vars = required_vars or []
    alternative_groups = alternative_groups or []
    ds_vars = set(ds.data_vars)

    # Missing mandatory variables
    missing_vars = [v for v in required_vars if v not in ds_vars]

    # Missing groups (at least one of the group required)
    missing_groups = []
    missing_groups.extend(
        group
        for group in alternative_groups
        if all(v not in ds_vars for v in group)
    )
    identical = not missing_vars and not missing_groups

    report = {
        "missing_vars": missing_vars,
        "missing_groups": missing_groups,
        "identical": identical,
    }

    if raise_error and not identical:
        lines = [f"Variable validation failed for {name}:"]
        if missing_vars:
            lines.append("  ❌ Missing required variables:")
            lines.extend(f"      - {v}" for v in missing_vars)
        if missing_groups:
            lines.append("  ❌ Missing one of the required alternatives:")
            lines.extend("      - One of: " + ", ".join(group) for group in missing_groups)
        raise ValueError("\n".join(lines))
    print(f"Variable validation for {name}: {'✅ PASSED' if identical else '❌ FAILED'}")
    return report


def validate_coords(
    ds,
    required_coords=None,
    *,
    name="dataset",
    raise_error=True
):
    """
    Validate that required coordinate variables exist and match the expected
    dimension shape.

    Parameters
    ----------
    ds : xr.Dataset
    required_coords : dict
        Mapping: coord_name -> expected_dimension_name
        Example: {"lat": "station", "lon": "station"}
    name : str
    raise_error : bool

    Returns
    -------
    dict with:
        - missing_coords
        - wrong_dim_coords
        - identical (bool)
    """
    required_coords = required_coords or {}

    missing = []
    wrong_dim = {}

    for coord, expected_dim in required_coords.items():
        if coord not in ds.coords:
            missing.append(coord)
            continue

        dims = ds[coord].dims
        if len(dims) != 1 or dims[0] != expected_dim:
            wrong_dim[coord] = {
                "actual": dims,
                "expected": (expected_dim,),
            }

    identical = not missing and not wrong_dim

    report = {
        "missing_coords": missing,
        "wrong_dim_coords": wrong_dim,
        "identical": identical,
    }

    if raise_error and not identical:
        lines = [f"Coordinate validation failed for {name}:"]
        if missing:
            lines.append("  ❌ Missing required coordinates:")
            lines.extend(f"      - {c}" for c in missing)
        if wrong_dim:
            lines.append("  ❌ Coordinates with incorrect dimensions:")
            lines.extend(
                f"      - {c}: actual dims {info['actual']} (expected {info['expected']})"
                for c, info in wrong_dim.items()
            )
        raise ValueError("\n".join(lines))
    print(f"Coordinate validation for {name}: {'✅ PASSED' if identical else '❌ FAILED'}")
    return report


import warnings

def validate_units(ds, expected_units):
    """
    Validate that key variables and coordinates have expected units.
    Case-insensitive matching. Warns on mismatch.
    """

    for var, acceptable_units in expected_units.items():
        if var not in ds.variables:
            continue  # other validators cover missing vars

        actual = ds[var].attrs.get("units", None)

        # Missing units attribute
        if actual is None:
            warnings.warn(
                f"⚠️ [units] {var} is missing a 'units' attribute "
                f"(expected: {acceptable_units})"
            )
            continue

        # Case-insensitive comparison
        actual_norm = actual.lower()
        expected_norm = [u.lower() for u in acceptable_units]

        if actual_norm not in expected_norm:
            warnings.warn(
                f"⚠️ [units] {var} has units '{actual}', "
                f"but expected one of {acceptable_units}"
            )


# Use the validator but don't raise an exception so the cell returns a report instead of erroring.
validate_dims(ds, correct_dimensions, name="ds", raise_error=True)
validate_vars(
    ds,
    required_vars=required_vars,
    alternative_groups=alternative_groups,
    name="pressure profile DSD",
)
validate_coords(
    ds,
    required_coords=required_coords,
    name="pressure profile DSD",
)
validate_units(ds, expected_units)

Dimension validation for ds: ✅ PASSED
Variable validation for pressure profile DSD: ✅ PASSED
Coordinate validation for pressure profile DSD: ✅ PASSED
